# Proyecto de Optimizacion sobre Grafos

Este notebook resuelve el caso planteado en `Enunciado.pdf` usando una red dirigida construida a partir de `matriz_de_datos2.csv`.

La idea general es trabajar la misma red desde tres enfoques:

- `Flujo al costo minimo`: enviar 600 unidades minimizando el costo total.
- `Flujo maximo`: encontrar cuanto flujo total soporta la red.
- `Ruta mas corta`: encontrar la menor distancia entre los origenes 1 o 2 y los destinos 78, 79 y 80.

Antes de cada modelo se hace un cambio manual sobre la red para analizar sensibilidad y observar si la solucion mejora.

In [ ]:
# Librerias base para leer la matriz de datos y apoyar algunos calculos.
# Todo el proyecto parte del CSV y luego construye un grafo dirigido.
import pandas as pd
import numpy as np

In [ ]:
# Carga de datos del caso.
# Cada fila del archivo representa un arco Origen -> Destino
# con tres atributos: Costo, Distancia y Capacidad.
url = "https://raw.githubusercontent.com/Samu-Kiss/UNI-26-10-Proyecto-Optimizacion/refs/heads/main/matriz_de_datos2.csv"
df = pd.read_csv(url)
df.head(100)

## 1. Carga de datos y construccion del grafo

En este bloque se leen los datos del caso, se limpian las columnas y se construye el grafo dirigido base.

Tambien se identifican automaticamente:

- `Fuentes`: nodos sin arcos de entrada.
- `Sumideros`: nodos sin arcos de salida.
- `Intermedios`: nodos que conectan el flujo entre fuentes y destinos.

La visualizacion por columnas sirve para entender la estructura de la red antes de optimizar.

In [ ]:
# Construccion y visualizacion del grafo base del caso.
# En esta celda se prepara la red, se identifican fuentes y sumideros,
# y se genera un dibujo por columnas para entender la estructura del problema.
import networkx as nx
from collections import defaultdict, deque
from html import escape
from IPython.display import HTML, display

# Limpia nombres de columnas para evitar espacios en el encabezado del CSV.
df.columns = df.columns.str.strip()

# Convierte columnas relevantes a numericas por seguridad.
for col in ["Origen", "Destino", "Costo", "Distancia", "Capacidad"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=["Origen", "Destino"])
df[["Origen", "Destino"]] = df[["Origen", "Destino"]].astype(int)

# Crea un grafo dirigido a partir de las columnas Origen y Destino.
G = nx.from_pandas_edgelist(
    df,
    source="Origen",
    target="Destino",
    edge_attr=["Costo", "Distancia", "Capacidad"],
    create_using=nx.DiGraph(),
)

# Clasificacion automatica de nodos segun su rol dentro de la red.
nodes = sorted(G.nodes())
fuentes = sorted([n for n in nodes if G.in_degree(n) == 0])
sumideros = sorted([n for n in nodes if G.out_degree(n) == 0])

# Respaldo para casos no estandar (por ejemplo, grafos con ciclos sin fuente/sumidero claro).
if not fuentes and nodes:
    fuentes = [nodes[0]]
if not sumideros and nodes:
    sumideros = [nodes[-1]]

set_fuentes = set(fuentes)
set_sumideros = set(sumideros)
intermedios = [n for n in nodes if n not in set_fuentes and n not in set_sumideros]

# Nivel inicial por cantidad minima de saltos desde las fuentes.
# Esto se usa solo para dibujar el grafo de izquierda a derecha.
niveles = {n: 0 for n in fuentes}
cola = deque(fuentes)
while cola:
    u = cola.popleft()
    for v in G.successors(u):
        nuevo_nivel = niveles[u] + 1
        if v not in niveles or nuevo_nivel < niveles[v]:
            niveles[v] = nuevo_nivel
            cola.append(v)

# Asignacion inicial de columnas.
columna_nodo = {}
for n in fuentes:
    columna_nodo[n] = 0
for n in intermedios:
    columna_nodo[n] = max(1, niveles.get(n, 1))

max_col_intermedio = max((columna_nodo[n] for n in intermedios), default=0)
col_sumidero = max_col_intermedio + 1
for n in sumideros:
    columna_nodo[n] = col_sumidero

# Regla pedida: si hay arco entre dos nodos, no pueden quedar en la misma columna.
max_iter = max(1, len(nodes) * len(nodes))
for _ in range(max_iter):
    changed = False
    for u, v in G.edges():
        if columna_nodo[u] == columna_nodo[v]:
            if v in set_sumideros:
                col_sumidero += 1
                for s in sumideros:
                    columna_nodo[s] = col_sumidero
            elif v in set_fuentes:
                if u not in set_fuentes and columna_nodo[u] > 0:
                    columna_nodo[u] -= 1
                else:
                    columna_nodo[v] += 1
            else:
                columna_nodo[v] += 1
            changed = True
    if not changed:
        break

# Compacta indices de columnas sin alterar el orden izquierda->derecha.
columnas_unicas = sorted(set(columna_nodo.values()))
remap_col = {c: i for i, c in enumerate(columnas_unicas)}
columna_nodo = {n: remap_col[c] for n, c in columna_nodo.items()}
col_sumidero = max((columna_nodo[s] for s in sumideros), default=0)

columnas = defaultdict(list)
for n, c in columna_nodo.items():
    columnas[c].append(n)


def y_positions(n, sep=1.0):
    if n == 1:
        return [0.0]
    centro = (n - 1) / 2
    return [(centro - i) * sep for i in range(n)]


# Reordena nodos dentro de cada columna para reducir cruces de aristas.
# Este bloque mejora la lectura visual, no cambia los modelos de optimizacion.
orden_col = {c: sorted(v, reverse=True) for c, v in columnas.items()}
columnas_ordenadas = sorted(orden_col.keys())
node_to_col = dict(columna_nodo)

for _ in range(5):
    pos_y = {}
    for c in columnas_ordenadas:
        for n, y in zip(orden_col[c], y_positions(len(orden_col[c]), sep=1.0)):
            pos_y[n] = y

    for c in columnas_ordenadas[1:]:
        def key_izq(nodo):
            vecinos = [
                pos_y[p]
                for p in G.predecessors(nodo)
                if node_to_col.get(p, c) < c and p in pos_y
            ]
            return sum(vecinos) / len(vecinos) if vecinos else pos_y.get(nodo, 0.0)

        orden_col[c] = sorted(orden_col[c], key=key_izq)

    pos_y = {}
    for c in columnas_ordenadas:
        for n, y in zip(orden_col[c], y_positions(len(orden_col[c]), sep=1.0)):
            pos_y[n] = y

    for c in reversed(columnas_ordenadas[:-1]):
        def key_der(nodo):
            vecinos = [
                pos_y[s]
                for s in G.successors(nodo)
                if node_to_col.get(s, c) > c and s in pos_y
            ]
            return sum(vecinos) / len(vecinos) if vecinos else pos_y.get(nodo, 0.0)

        orden_col[c] = sorted(orden_col[c], key=key_der)

# Orden final dentro de cada columna: de mayor a menor.
for c in columnas_ordenadas:
    orden_col[c] = sorted(orden_col[c], reverse=False)


# Alto maximo por columna = 14 unidades.
max_col_height = 14.0


def y_positions_limited(n, max_height=14.0, pref_sep=0.8):
    if n <= 1:
        return [0.0]
    sep = min(pref_sep, max_height / (n - 1))
    return [((n - 1) / 2 - i) * sep for i in range(n)]


x_sep = 2.8
pos = {}
for c in columnas_ordenadas:
    nodos_col = orden_col[c]
    ys = y_positions_limited(len(nodos_col), max_height=max_col_height, pref_sep=0.8)
    for nodo, y in zip(nodos_col, ys):
        pos[nodo] = (c * x_sep, y)

# Garantiza posicion de todos los nodos.
for n in nodes:
    if n not in pos:
        pos[n] = (x_sep, 0.0)

max_nodes_col = max((len(v) for v in columnas.values()), default=1)
node_size = int(max(170, min(480, 15000 / max_nodes_col)))
label_font = 7 if max_nodes_col >= 22 else 8

fig_w = max(15, (len(columnas_ordenadas) + 1) * 2.5)
fig_h = max(7, min(10, 4.8 + max_col_height * 0.24))

base_node_styles = {
    "Fuentes": {
        "nodes": fuentes,
        "color": "#8ecae6",
        "edge": "#1d3557",
    },
    "Intermedios": {
        "nodes": intermedios,
        "color": "#fde2e4",
        "edge": "#7b2cbf",
    },
    "Sumideros": {
        "nodes": sumideros,
        "color": "#caffbf",
        "edge": "#2a9d8f",
    },
}


def formatear_valor(valor):
    if isinstance(valor, float):
        if abs(valor - round(valor)) < 1e-9:
            return str(int(round(valor)))
        return f"{valor:.2f}"
    if isinstance(valor, int):
        return str(valor)
    return str(valor)


def tooltip_text(titulo, detalles):
    lineas = [str(titulo)]
    for clave, valor in detalles.items():
        if valor is None:
            continue
        lineas.append(f"{clave}: {formatear_valor(valor)}")
    return escape("\n".join(lineas))


def construir_metricas_flujo(flujos):
    edge_metrics = {}
    total_entrada = {n: 0.0 for n in G.nodes()}
    total_salida = {n: 0.0 for n in G.nodes()}

    for (u, v), valor in flujos.items():
        if valor is None or abs(valor) <= 1e-9 or not G.has_edge(u, v):
            continue
        edge_metrics[(u, v)] = {"Flujo": valor}
        total_salida[u] += float(valor)
        total_entrada[v] += float(valor)

    node_metrics = {}
    for n in G.nodes():
        if abs(total_entrada[n]) <= 1e-9 and abs(total_salida[n]) <= 1e-9:
            continue
        if n in set_fuentes:
            node_metrics[n] = {"Flujo que sale": total_salida[n]}
        elif n in set_sumideros:
            node_metrics[n] = {"Flujo que llega": total_entrada[n]}
        else:
            node_metrics[n] = {
                "Flujo que entra": total_entrada[n],
                "Flujo que sale": total_salida[n],
            }

    return edge_metrics, node_metrics


def plot_grafo_resaltado(
    highlighted_edges=None,
    highlighted_nodes=None,
    title="Grafo dirigido por columnas",
    subtitle=None,
    edge_metrics=None,
    node_metrics=None,
):
    highlighted_edges = list(dict.fromkeys(highlighted_edges or []))
    highlighted_nodes = set(highlighted_nodes or [])
    highlighted_edge_set = set(highlighted_edges)
    edge_metrics = edge_metrics or {}
    node_metrics = node_metrics or {}

    for u, v in highlighted_edges:
        highlighted_nodes.update([u, v])

    xs = [xy[0] for xy in pos.values()]
    ys = [xy[1] for xy in pos.values()]
    min_x, max_x = min(xs), max(xs)
    min_y, max_y = min(ys), max(ys)

    svg_width = max(1120, int(fig_w * 95))
    svg_height = max(560, int(fig_h * 95))
    margin_x = 115
    margin_y = 70

    span_x = max(1e-6, max_x - min_x)
    span_y = max(1e-6, max_y - min_y)

    def sx(x):
        return margin_x + (x - min_x) * (svg_width - 2 * margin_x) / span_x

    def sy(y):
        return margin_y + (max_y - y) * (svg_height - 2 * margin_y) / span_y

    svg_pos = {n: (sx(x), sy(y)) for n, (x, y) in pos.items()}
    node_radius = max(13, min(20, int(node_size / 18)))
    label_size = max(11, label_font + 4)

    legend_parts = [
        '<span style="display:inline-flex;align-items:center;gap:8px;"><span style="width:14px;height:14px;border-radius:999px;background:#8ecae6;border:2px solid #1d3557;display:inline-block;"></span>Fuentes</span>',
        '<span style="display:inline-flex;align-items:center;gap:8px;"><span style="width:14px;height:14px;border-radius:999px;background:#fde2e4;border:2px solid #7b2cbf;display:inline-block;"></span>Intermedios</span>',
        '<span style="display:inline-flex;align-items:center;gap:8px;"><span style="width:14px;height:14px;border-radius:999px;background:#caffbf;border:2px solid #2a9d8f;display:inline-block;"></span>Sumideros</span>',
    ]
    if highlighted_edges:
        legend_parts.append(
            '<span style="display:inline-flex;align-items:center;gap:8px;"><span style="width:28px;height:0;border-top:3px solid #d62828;display:inline-block;"></span>Aristas resaltadas</span>'
        )

    svg_parts = [
        f'<div style="max-width:{svg_width}px;margin:18px auto;padding:18px 18px 10px 18px;background:linear-gradient(180deg,#fbf8f1 0%,#f3ecdf 100%);border:1px solid #e2d7c3;border-radius:24px;box-shadow:0 14px 34px rgba(80,56,34,0.08);">',
        '<div style="text-align:center;font-family:Segoe UI,Arial,sans-serif;margin-bottom:12px;">',
        f'<div style="font-size:20px;font-weight:700;color:#4a3426;">{escape(title)}</div>',
    ]

    if subtitle:
        svg_parts.append(
            f'<div style="font-size:13px;color:#7a5b45;margin-top:5px;">{escape(subtitle)}</div>'
        )

    svg_parts.append(
        f'<div style="margin-top:10px;display:flex;justify-content:center;gap:18px;flex-wrap:wrap;font-size:12px;color:#5f4634;">{"".join(legend_parts)}</div>'
    )
    svg_parts.append('</div>')
    svg_parts.append(
        f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 {svg_width} {svg_height}" style="width:100%;height:auto;display:block;">'
    )
    svg_parts.append(
        '<defs>'
        '<linearGradient id="bg-gradient" x1="0" y1="0" x2="0" y2="1">'
        '<stop offset="0%" stop-color="#fbf8f1"/>'
        '<stop offset="100%" stop-color="#efe5d6"/>'
        '</linearGradient>'
        '<marker id="arrow-base" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse">'
        '<path d="M 0 0 L 10 5 L 0 10 z" fill="#9b9b9b" fill-opacity="0.55"/>'
        '</marker>'
        '<marker id="arrow-highlight" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="8" markerHeight="8" orient="auto-start-reverse">'
        '<path d="M 0 0 L 10 5 L 0 10 z" fill="#d62828"/>'
        '</marker>'
        '</defs>'
    )
    svg_parts.append(
        f'<rect x="0" y="0" width="{svg_width}" height="{svg_height}" rx="22" ry="22" fill="url(#bg-gradient)" />'
    )

    for u, v, data in G.edges(data=True):
        x1, y1 = svg_pos[u]
        x2, y2 = svg_pos[v]
        is_highlighted = (u, v) in highlighted_edge_set
        color = '#d62828' if is_highlighted else '#8a8a8a'
        marker = 'url(#arrow-highlight)' if is_highlighted else 'url(#arrow-base)'
        stroke_width = 3.2 if is_highlighted else 1.1
        opacity = 0.93 if is_highlighted else 0.23

        detalles_arco = {
            'Origen': u,
            'Destino': v,
            'Costo': data.get('Costo'),
            'Distancia': data.get('Distancia'),
            'Capacidad': data.get('Capacidad'),
        }
        detalles_arco.update(edge_metrics.get((u, v), {}))
        tooltip_arco = tooltip_text(f'Arco {u} -> {v}', detalles_arco)

        svg_parts.append(
            f'<g>'
            f'<line x1="{x1:.1f}" y1="{y1:.1f}" x2="{x2:.1f}" y2="{y2:.1f}" stroke="{color}" stroke-width="{stroke_width}" stroke-linecap="round" opacity="{opacity}" marker-end="{marker}" />'
            f'<line x1="{x1:.1f}" y1="{y1:.1f}" x2="{x2:.1f}" y2="{y2:.1f}" stroke="#000000" stroke-opacity="0.001" stroke-width="{max(10.0, stroke_width + 8):.1f}" stroke-linecap="round" pointer-events="stroke">'
            f'<title>{tooltip_arco}</title>'
            '</line>'
            '</g>'
        )

    for categoria, style in base_node_styles.items():
        for nodo in style['nodes']:
            cx, cy = svg_pos[nodo]
            is_highlighted = nodo in highlighted_nodes
            borde = '#d62828' if is_highlighted else style['edge']
            grosor = 3.2 if is_highlighted else 1.5
            color_texto = '#7f0000' if is_highlighted else '#1b1b1b'
            peso_texto = '700' if is_highlighted else '600'

            detalles_nodo = {'Rol': categoria, 'Nodo': nodo}
            detalles_nodo.update(node_metrics.get(nodo, {}))
            tooltip_nodo = tooltip_text(f'Nodo {nodo}', detalles_nodo)

            svg_parts.append(
                f'<g>'
                f'<circle cx="{cx:.1f}" cy="{cy:.1f}" r="{node_radius}" fill="{style["color"]}" stroke="{borde}" stroke-width="{grosor}">'
                f'<title>{tooltip_nodo}</title>'
                '</circle>'
                f'<text x="{cx:.1f}" y="{cy + 0.5:.1f}" text-anchor="middle" dominant-baseline="middle" font-family="Segoe UI,Arial,sans-serif" font-size="{label_size}" font-weight="{peso_texto}" fill="{color_texto}">{nodo}</text>'
            )

            etiqueta_visible = None
            anchor = 'middle'
            label_x = cx

            if nodo in set_fuentes and 'Flujo que sale' in node_metrics.get(nodo, {}):
                etiqueta_visible = f"sale: {formatear_valor(node_metrics[nodo]['Flujo que sale'])}"
                label_x = cx - node_radius - 10
                anchor = 'end'
            elif nodo in set_sumideros and 'Flujo que llega' in node_metrics.get(nodo, {}):
                etiqueta_visible = f"llega: {formatear_valor(node_metrics[nodo]['Flujo que llega'])}"
                label_x = cx + node_radius + 10
                anchor = 'start'

            if etiqueta_visible:
                svg_parts.append(
                    f'<text x="{label_x:.1f}" y="{cy + 4:.1f}" text-anchor="{anchor}" dominant-baseline="middle" font-family="Segoe UI,Arial,sans-serif" font-size="11" font-weight="700" fill="#7f0000">{escape(etiqueta_visible)}</text>'
                )

            svg_parts.append('</g>')

    svg_parts.append('</svg>')
    svg_parts.append('</div>')
    display(HTML(''.join(svg_parts)))


# Dibujo inicial del grafo base.
plot_grafo_resaltado(
    title="Grafo dirigido por columnas",
    subtitle="Pasa el mouse por cada arco para ver costo, distancia y capacidad.",
)


# 2. Flujo al costo minimo

En esta seccion se modifica primero un costo de la red y luego se resuelve el modelo de costo minimo.

El objetivo es enviar exactamente `600` unidades con dos condiciones:

- respetar la capacidad de cada arco;
- garantizar que al menos `120` unidades lleguen al nodo `80`.

El cambio manual analizado aqui es sobre el arco `62 -> 78`.

### ATENCIÓN: NO CORRER EL SCRIPT A CONTINUACIÓN SI NO SE QUIERE HACER LA MODIFICACION PROPUESTA AÚN

In [ ]:
# ANALISIS DE SENSIBILIDAD PARA EL MODELO DE COSTO MINIMO
# Cambio manual aplicado: reducir el costo del arco 62 -> 78.
# En la fuente actual el costo pasa de 14 a 5 para probar si el enlace
# se vuelve mas atractivo dentro del modelo de flujo al costo minimo.
# Nota: la salida guardada del notebook muestra 4 porque fue ejecutado
# antes con ese valor; si se vuelve a correr esta version, usara 5.

print("Costo original arco 62->78:", G[62][78]["Costo"])

G[62][78]["Costo"] = 5

print("Nuevo costo arco 62->78:", G[62][78]["Costo"])

### Interpretacion de la solucion de costo minimo

La siguiente celda imprime solamente los arcos con flujo positivo.

Eso permite ver por donde decide pasar el modelo una vez minimiza el costo total. En otras palabras, muestra la red efectiva que usa la solucion optima.

In [ ]:
# MODELO 1: FLUJO DE COSTO MINIMO
# Objetivo: enviar 600 unidades por la red al menor costo total posible.
# Restricciones principales:
# 1. Cada arco no puede superar su capacidad.
# 2. En nodos intermedios se conserva el flujo.
# 3. El total enviado desde las fuentes debe ser exactamente 600.
# 4. Al menos 20% del flujo total (120 unidades) debe llegar al nodo 80.
import pulp

# Crear modelo
model = pulp.LpProblem("MinCostFlow", pulp.LpMinimize)

# Variables de decision: flujo que viaja por cada arco (u, v).
flow = {}

for u, v, data in G.edges(data=True):
    cap = data.get("Capacidad", 999999)
    flow[(u,v)] = pulp.LpVariable(f"f_{u}_{v}", lowBound=0, upBound=cap)

# Funcion objetivo: minimizar el costo total acumulado en toda la red.
model += pulp.lpSum(
    flow[(u,v)] * data["Costo"]
    for u, v, data in G.edges(data=True)
)

# -------------------------
# CONSERVACION DE FLUJO
# -------------------------

for n in G.nodes():

    inflow = pulp.lpSum(flow[(u,n)] for u,n2 in flow if n2 == n)
    outflow = pulp.lpSum(flow[(n,v)] for n2,v in flow if n2 == n)

    # En fuentes se permite sacar flujo hasta completar el total pedido.
    if n in fuentes:
        model += outflow - inflow <= 600

    # En sumideros se permite recibir flujo neto.
    elif n in sumideros:
        model += inflow - outflow >= 0

    # En nodos intermedios: lo que entra debe salir.
    else:
        model += inflow == outflow

# -------------------------
# TOTAL ENVIADO
# -------------------------

# El problema exige enviar exactamente 600 unidades en total.
model += pulp.lpSum(
    flow[(u,v)] for u,v in flow if u in fuentes
) == 600

# -------------------------
# 20% minimo al nodo 80
# -------------------------

# El 20% de 600 equivale a 120 unidades.
model += pulp.lpSum(
    flow[(u,80)] for u,v in flow if v == 80
) >= 120

# Resolver el modelo con CBC usando explicitamente el metodo simplex.
solver_simplex = pulp.PULP_CBC_CMD(msg=True, mip=False, options=["primalSimplex"])
model.solve(solver_simplex)

print("Status:", pulp.LpStatus[model.status])
print("Costo mínimo:", pulp.value(model.objective))
# Impresion de los arcos que realmente quedan activos en la solucion optima.
# Solo se muestran los enlaces con flujo positivo.
for (u,v), var in flow.items():
    if var.value() > 0:
        print(f"{u} -> {v} : {var.value()}")

# Visualizacion de la solucion: se resaltan en rojo los arcos usados.
flujos_minimos = {
    (u, v): var.value()
    for (u, v), var in flow.items()
    if var.value() is not None and var.value() > 0
}
arcos_flujo_minimo = list(flujos_minimos.keys())
metricas_arcos_minimo, metricas_nodos_minimo = construir_metricas_flujo(flujos_minimos)

plot_grafo_resaltado(
    highlighted_edges=arcos_flujo_minimo,
    title="Flujo al costo minimo (Simplex): red utilizada",
    subtitle="Pasa el mouse por cada arco para ver costo, capacidad y flujo. En fuentes y sumideros se muestra el flujo total.",
    edge_metrics=metricas_arcos_minimo,
    node_metrics=metricas_nodos_minimo,
)


## 3. Flujo maximo

En esta parte ya no se usa el atributo `Costo`. Lo importante es la capacidad total de transporte de la red.

Primero se cambian manualmente algunas capacidades y luego se calcula el flujo maximo total desde todas las fuentes hasta todos los sumideros.

Para resolverlo de forma estandar se crea:

- una `superfuente S`, conectada a todas las fuentes;
- un `supersumidero T`, conectado a todos los destinos finales.

### ATENCIÓN: NO CORRER EL SCRIPT A CONTINUACIÓN SI NO SE QUIERE HACER LA MODIFICACION PROPUESTA AÚN

In [ ]:
# ANALISIS DE SENSIBILIDAD PARA EL MODELO DE FLUJO MAXIMO
# Cambios manuales aplicados sobre capacidades:
# 63 -> 46: 26 a 150
# 63 -> 17: 22 a 150
# 23 -> 59: 44 a 200
# La idea es probar si ampliar estos enlaces aumenta el flujo total de la red.

print("Capacidad original 63->46:", G[63][46]["Capacidad"])
print("Capacidad original 63->17:", G[63][17]["Capacidad"])
print("Capacidad original 23->59:", G[23][59]["Capacidad"])

G[63][46]["Capacidad"] = 150
G[63][17]["Capacidad"] = 150
G[23][59]["Capacidad"] = 200


print("Nueva capacidad 63->46:", G[63][46]["Capacidad"])
print("Nueva capacidad 63->17:", G[63][17]["Capacidad"])
print("Nueva capacidad 23->59:", G[23][59]["Capacidad"])

### Interpretacion de la solucion de flujo maximo

La salida del modelo reporta dos cosas:

- el valor total del flujo maximo alcanzado;
- los arcos por los que efectivamente circula flujo en esa solucion.

Esto permite ver no solo cuanto soporta la red, sino tambien como se distribuye ese flujo.

In [ ]:
# MODELO 2: FLUJO MAXIMO
# En este modelo ya no importa el costo: solo interesa cuanto flujo total
# puede atravesar la red desde todas las fuentes hasta todos los sumideros.
# Para manejar multiples origenes y destinos se crea una superfuente S
# y un supersumidero T.
import networkx as nx
from math import inf


def ford_fulkerson_manual(graph, source, sink):
    residual = nx.DiGraph()

    for u, v, data in graph.edges(data=True):
        capacidad = data.get("capacity", 0)
        residual.add_edge(u, v, capacity=capacidad)
        if not residual.has_edge(v, u):
            residual.add_edge(v, u, capacity=0)

    max_flow = 0

    while True:
        parent = {source: None}
        stack = [source]

        while stack and sink not in parent:
            u = stack.pop()
            for v in residual.successors(u):
                if v not in parent and residual[u][v]["capacity"] > 0:
                    parent[v] = u
                    stack.append(v)

        if sink not in parent:
            break

        path_flow = inf
        v = sink
        while parent[v] is not None:
            u = parent[v]
            path_flow = min(path_flow, residual[u][v]["capacity"])
            v = u

        v = sink
        while parent[v] is not None:
            u = parent[v]
            residual[u][v]["capacity"] -= path_flow
            residual[v][u]["capacity"] += path_flow
            v = u

        max_flow += path_flow

    flow_dict = {u: {} for u in graph.nodes()}
    for u, v, data in graph.edges(data=True):
        capacidad_original = data.get("capacity", 0)
        capacidad_residual = residual[u][v]["capacity"] if residual.has_edge(u, v) else 0
        flow_dict[u][v] = capacidad_original - capacidad_residual

    return max_flow, flow_dict

# Crear un grafo que conserve unicamente las capacidades.
G_cap = nx.DiGraph()

for u,v,data in G.edges(data=True):
    G_cap.add_edge(u, v, capacity=data["Capacidad"])

# Super fuente
G_cap.add_node("S")

for f in fuentes:
    G_cap.add_edge("S", f, capacity=999999)

# Super sumidero
G_cap.add_node("T")

for s in sumideros:
    G_cap.add_edge(s, "T", capacity=999999)

# Resolver el problema de flujo maximo entre S y T con Ford-Fulkerson.
flow_value, flow_dict = ford_fulkerson_manual(G_cap, "S", "T")

print("Flujo máximo:", flow_value)
print("\nArcos con flujo:")

# Mostrar unicamente los arcos por los que efectivamente pasa flujo.
for u in flow_dict:
    for v in flow_dict[u]:
        if flow_dict[u][v] > 0:
            print(f"{u} -> {v}  flujo: {flow_dict[u][v]}")

# Visualizacion de la solucion de flujo maximo.
# Se excluyen las conexiones artificiales S y T para dibujar solo el grafo original.
flujos_maximos = {
    (u, v): flow_dict[u][v]
    for u in flow_dict
    for v in flow_dict[u]
    if flow_dict[u][v] > 0 and u in G.nodes() and v in G.nodes()
}
arcos_flujo_maximo = list(flujos_maximos.keys())
metricas_arcos_maximo, metricas_nodos_maximo = construir_metricas_flujo(flujos_maximos)

plot_grafo_resaltado(
    highlighted_edges=arcos_flujo_maximo,
    title="Flujo maximo (Ford-Fulkerson): arcos utilizados",
    subtitle=f"Flujo total alcanzado = {flow_value}. Pasa el mouse por cada arco para ver costo, capacidad y flujo.",
    edge_metrics=metricas_arcos_maximo,
    node_metrics=metricas_nodos_maximo,
)


## 4. Ruta mas corta

En esta seccion se trabaja con el atributo `Distancia` para encontrar las rutas mas cortas desde el origen `1` hacia los destinos:

- `78`;
- `79`;
- `80`.

Primero se hace un cambio manual en la distancia del arco `6 -> 14` y luego se calcula la mejor ruta para cada destino por separado.

### ATENCIÓN: NO CORRER EL SCRIPT A CONTINUACIÓN SI NO SE QUIERE HACER LA MODIFICACION PROPUESTA AÚN

In [ ]:
# ANALISIS DE SENSIBILIDAD PARA EL MODELO DE RUTA MAS CORTA
# Cambio manual aplicado: reducir la distancia del arco 6 -> 14 de 32 a 15.
# La idea es probar si ese enlace mejora las rutas desde el origen 1
# hacia los destinos 78, 79 y 80.

print("Distancia original arco 6->14:", G[6][14]["Distancia"])

G[6][14]["Distancia"] = 15

print("Nueva distancia arco 6->14:", G[6][14]["Distancia"])

### Rutas por destino y visualizacion final

Primero se imprimen las rutas mas cortas desde el nodo `1` hacia cada destino. Despues, en la ultima celda, se muestra una visualizacion independiente para `78`, `79` y `80`.

Asi no se pierde ninguna de las tres respuestas que pide el analisis.

In [ ]:
# MODELO 3: RUTA MAS CORTA
# Se calcula la ruta de menor distancia desde el origen 1
# hacia cada uno de los destinos 78, 79 y 80.
origen = 1
destinos = [78,79,80]
rutas_por_destino = {}
distancias_por_destino = {}

for d in destinos:

    try:
        # Encontrar la mejor ruta segun el atributo Distancia con Dijkstra.
        path = nx.dijkstra_path(G, origen, d, weight="Distancia")
        dist = nx.dijkstra_path_length(G, origen, d, weight="Distancia")

        rutas_por_destino[d] = path
        distancias_por_destino[d] = dist

        print(f"Origen -> Destino:  {origen} -> {d}")
        print("Ruta:", path)
        print("Distancia:", dist)
        print()

    except nx.NetworkXNoPath:
        print(f"No hay ruta {origen} -> {d}")
        # Visualizar por separado la mejor ruta desde 1 hacia cada destino.
for destino in destinos:
    if destino not in rutas_por_destino:
        continue

    ruta_actual = rutas_por_destino[destino]
    distancia_actual = distancias_por_destino[destino]
    arcos_ruta_corta = list(zip(ruta_actual[:-1], ruta_actual[1:]))
    metricas_arcos_ruta = {(u, v): {"En la ruta": "Si"} for (u, v) in arcos_ruta_corta}
    metricas_nodos_ruta = {}

    for posicion, nodo in enumerate(ruta_actual):
        if posicion == 0:
            metricas_nodos_ruta[nodo] = {"Rol en la ruta": "Origen"}
        elif posicion == len(ruta_actual) - 1:
            metricas_nodos_ruta[nodo] = {"Rol en la ruta": "Destino"}
        else:
            metricas_nodos_ruta[nodo] = {"Rol en la ruta": "Paso intermedio"}

    print("Ruta más corta encontrada:")
    print(f"Origen -> Destino: {origen} -> {destino}")
    print("Ruta:", ruta_actual)
    print("Distancia total:", distancia_actual)
    print()

    plot_grafo_resaltado(
        highlighted_edges=arcos_ruta_corta,
        highlighted_nodes=ruta_actual,
        title=f"Ruta mas corta (Dijkstra): {origen} -> {destino}",
        subtitle=f"Distancia total = {distancia_actual}. Pasa el mouse por cada arco para ver costo, distancia y capacidad.",
        edge_metrics=metricas_arcos_ruta,
        node_metrics=metricas_nodos_ruta,
    )
